In [1]:
Cell 1

SyntaxError: invalid syntax (1472947084.py, line 1)

In [2]:
import pandas as pd
pd.set_option("display.max_columns", 50)
df = pd.read_csv("../data/raw/Leads.csv")
print(df.shape)
df.head()

(9240, 37)


,Prospect ID,Lead Number,Lead Origin,Lead Source,Do Not Email,Do Not Call,Converted,TotalVisits,Total Time Spent on Website,Page Views Per Visit,Last Activity,Country,Specialization,How did you hear about X Education,What is your current occupation,What matters most to you in choosing a course,Search,Magazine,Newspaper Article,X Education Forums,Newspaper,Digital Advertisement,Through Recommendations,Receive More Updates About Our Courses,Tags,Lead Quality,Update me on Supply Chain Content,Get updates on DM Content,Lead Profile,City,Asymmetrique Activity Index,Asymmetrique Profile Index,Asymmetrique Activity Score,Asymmetrique Profile Score,I agree to pay the amount through cheque,A free copy of Mastering The Interview,Last Notable Activity
0,7927b2df-8bba-4d29-b9a2-b6e0beafe620,660737,API,Olark Chat,No,No,0,0.0,0,0.0,Page Visited on Website,NaN,Select,Select,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Interested in other courses,Low in Relevance,No,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Modified
1,2a272436-5132-4136-86fa-dcc88c88f482,660728,API,Organic Search,No,No,0,5.0,674,2.5,Email Opened,India,Select,Select,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Ringing,NaN,No,No,Select,Select,02.Medium,02.Medium,15.0,15.0,No,No,Email Opened
2,8cc8c611-a219-4f35-ad23-fdfd2656bd8a,660727,Landing Page Submission,Direct Traffic,No,No,1,2.0,1532,2.0,Email Opened,India,Business Administration,Select,Student,Better Career Prospects,No,No,No,No,No,No,No,No,Will revert after reading the email,Might be,No,No,Potential Lead,Mumbai,02.Medium,01.High,14.0,20.0,No,Yes,Email Opened
3,0cc2df48-7cf4-4e39-9de9-19797f9b38cc,660719,Landing Page Submission,Direct Traffic,No,No,0,1.0,305,1.0,Unreachable,India,Media and Advertising,Word Of Mouth,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Ringing,Not Sure,No,No,Select,Mumbai,02.Medium,01.High,13.0,17.0,No,No,Modified
4,3256f628-e534-4826-9d63-4a8b88782852,660681,Landing Page Submission,Google,No,No,1,2.0,1428,1.0,Converted to Lead,India,Select,Other,Unemployed,Better Career Prospects,No,No,No,No,No,No,No,No,Will revert after reading the email,Might be,No,No,Select,Mumbai,02.Medium,01.High,15.0,18.0,No,No,Modified


In [3]:
print(df.columns.tolist())

['Prospect ID', 'Lead Number', 'Lead Origin', 'Lead Source', 'Do Not Email', 'Do Not Call', 'Converted', 'TotalVisits', 'Total Time Spent on Website', 'Page Views Per Visit', 'Last Activity', 'Country', 'Specialization', 'How did you hear about X Education', 'What is your current occupation', 'What matters most to you in choosing a course', 'Search', 'Magazine', 'Newspaper Article', 'X Education Forums', 'Newspaper', 'Digital Advertisement', 'Through Recommendations', 'Receive More Updates About Our Courses', 'Tags', 'Lead Quality', 'Update me on Supply Chain Content', 'Get updates on DM Content', 'Lead Profile', 'City', 'Asymmetrique Activity Index', 'Asymmetrique Profile Index', 'Asymmetrique Activity Score', 'Asymmetrique Profile Score', 'I agree to pay the amount through cheque', 'A free copy of Mastering The Interview', 'Last Notable Activity']


In [4]:
select_counts = (df == "Select").sum()
select_counts[select_counts > 0].sort_values(ascending=False)

How did you hear about X Education    5043
Lead Profile                          4146
City                                  2249
Specialization                        1942
dtype: int64

## My EDA findings – 8 Oct 2026

**Dataset:** Kaggle X Education Lead Scoring, 9,240 leads × 37 columns.

**Hidden missing values ("Select"):** found in 4 columns – How did you hear about X Education (54.6%), Lead Profile (44.9%), City (24.3%), Specialization (21.0%). "Select" is the form's default dropdown value, so it is converted to a missing value during ingestion.

**Conversion rate:** ___  (from Cell 5)

**Leakage check (Tags):** ___  (from Cell 7 – tags with conversion rate near 1.0 or 0.0)

**Decision:** behaviour fields (source, origin, visits, time on site, pages per visit, last activity, occupation, specialization) are the main model features. Sales-staff fields (Tags, Lead Quality, Lead Profile, Last Notable Activity) are used only in a leakage check.

In [5]:
(df.isna().mean() * 100).round(1).sort_values(ascending=False).head(15)

Lead Quality                                     51.6
Asymmetrique Profile Index                       45.6
Asymmetrique Activity Score                      45.6
Asymmetrique Activity Index                      45.6
Asymmetrique Profile Score                       45.6
Tags                                             36.3
What matters most to you in choosing a course    29.3
Lead Profile                                     29.3
What is your current occupation                  29.1
Country                                          26.6
How did you hear about X Education               23.9
Specialization                                   15.6
City                                             15.4
Page Views Per Visit                              1.5
TotalVisits                                       1.5
dtype: float64

In [6]:
print("Conversion rate:", round(df["Converted"].mean(), 3))
df["Converted"].value_counts()

Conversion rate: 0.385


Converted
0    5679
1    3561
Name: count, dtype: int64

In [7]:
print(df["Lead Source"].value_counts().head(10), "\n")
print(df["Last Activity"].value_counts(), "\n")
df[["TotalVisits", "Total Time Spent on Website", "Page Views Per Visit"]].describe().round(1)

Lead Source
Google              2868
Direct Traffic      2543
Olark Chat          1755
Organic Search      1154
Reference            534
Welingak Website     142
Referral Sites       125
Facebook              55
bing                   6
google                 5
Name: count, dtype: int64 

Last Activity
Email Opened                    3437
SMS Sent                        2745
Olark Chat Conversation          973
Page Visited on Website          640
Converted to Lead                428
Email Bounced                    326
Email Link Clicked               267
Form Submitted on Website        116
Unreachable                       93
Unsubscribed                      61
Had a Phone Conversation          30
Approached upfront                 9
View in browser link Clicked       6
Email Received                     2
Email Marked Spam                  2
Visited Booth in Tradeshow         1
Resubscribed to emails             1
Name: count, dtype: int64 



,TotalVisits,Total Time Spent on Website,Page Views Per Visit
count,9103.0,9240.0,9103.0
mean,3.4,487.7,2.4
std,4.9,548.0,2.2
min,0.0,0.0,0.0
25%,1.0,12.0,1.0
50%,3.0,248.0,2.0
75%,5.0,936.0,3.0
max,251.0,2272.0,55.0


In [8]:
tag_rate = df.groupby("Tags")["Converted"].agg(leads="size", conversion_rate="mean")
tag_rate.sort_values("conversion_rate", ascending=False).round(3)

,leads,conversion_rate
Tags,,
Interested in Next batch,5,1.000
Lateral student,3,1.000
Closed by Horizzon,358,0.994
Lost to EINS,175,0.977
Will revert after reading the email,2072,0.969
Busy,186,0.565
Shall take in the next coming month,2,0.500
Want to take admission but has financial problems,6,0.333
in touch with EINS,12,0.250


## EDA findings – C3-01 (8 Oct 2026, Shathursini J)

### 1. Dataset
- Kaggle "Lead Scoring Dataset" (X Education, amritachatterjee09): **9,240 leads × 37 columns**.
- One row = one lead; label = `Converted` (1 = became a customer, 0 = did not).

### 2. Conversion rate (label)
- **0.385** – 3,561 converted, 5,679 not converted.
- Higher than the "~30%" stated on Kaggle, so the measured value (38.5%) is used in all reporting.
- Mild class imbalance → **AUC and F1** are used instead of plain accuracy.

### 3. Hidden missing values ("Select")
- "Select" is the form's default dropdown value, meaning the person skipped the question.
- Found in 4 columns: How did you hear about X Education (5,043 = 54.6%), Lead Profile (4,146 = 44.9%), City (2,249 = 24.3%), Specialization (1,942 = 21.0%).
- **Decision:** "Select" is converted to a missing value during ingestion; the feature pipeline later fills missing categories with "Unknown".

### 4. Missing values (empty cells)
- Most incomplete: Lead Quality (51.6%), the four Asymmetrique index/score columns (45.6%), Tags (36.3%), Lead Profile (29.3%), occupation (29.1%), Country (26.6%).
- Behaviour fields are almost complete: TotalVisits and Page Views Per Visit (1.5%), Total Time Spent on Website (0%).

### 5. Behaviour fields
- Top lead sources: Google (2,868), Direct Traffic (2,543), Olark Chat (1,755), Organic Search (1,154), Reference (534).
- Top last activities: Email Opened (3,437), SMS Sent (2,745), Olark Chat Conversation (973).
- TotalVisits: median 3, max 251 → strong outliers. Page views per visit: median 2, max 55.
- Time on website: median 248 s, 25% of leads ≤ 12 s, max 2,272 s.

### 6. Data-quality issues found
- "Google" (2,868) and "google" (5) are the same source → merged during ingestion.
- Outliers in visits and page views → handled with a log transform in the feature pipeline.

### 7. Leakage check – Tags
- Several tags almost reveal the outcome:
  - Near 100% conversion: Closed by Horizzon (358 leads, 99.4%), Will revert after reading the email (2,072, 96.9%), Lost to EINS (175, 97.7%).
  - Near 0% conversion: Ringing (1,203, 2.8%), Already a student (465, 0.6%), Interested in other courses (513, 2.5%), switched off (240, 1.7%), wrong number given (47, 0%).
- These tags are written by the sales team during or after contact, so they are not available when a new lead arrives.
- "Lost to EINS" converting at 97.7% contradicts its name, which suggests inconsistent tagging.
- **Decision:** Tags, Lead Quality, Lead Profile and Last Notable Activity are **not used as main model features**. They are used only in a separate leakage check (C3-04) to show how much they inflate AUC.

### 8. Other uses of Tags
- Some tags describe **reasons a lead did not convert** (e.g. Interested in other courses, Not doing further education, Diploma holder (Not Eligible), financial problems). These will be used as an **objection proxy** in behavioural analytics (C3-10), not as model features.

### 9. Features chosen for the lead scoring model
- Numeric: TotalVisits, Total Time Spent on Website, Page Views Per Visit (+ time per visit, derived).
- Categorical: Lead Source, Lead Origin, Last Activity, current occupation, Specialization.
- Next step: ingestion script mapping these fields to a GHL-style lead schema split into 3 tenants (C3-01).